# ProwriteAI — Smart Text Improvement

**Projet NLP — Université Paris Dauphine, Campus Tunis — Robbana Amenallah — encadré par Mme Tarifa Amel — 2026-2027**

Objectif : transformer un message anglais informel (fautes, abréviations, ton familier) en un message correct et professionnel, sans changer son sens.

On compare quatre modèles Encoder–Decoder, tous **entraînés depuis zéro** en PyTorch (aucun modèle pré-entraîné, aucune API) :

| Modèle | Idée |
|---|---|
| `A_LSTM` | encodeur LSTM → état final → décodeur LSTM |
| `B_BiLSTM` | encodeur bidirectionnel + couche « pont » vers le décodeur |
| `C_Attention` | BiLSTM + attention de Bahdanau |
| `D_Pointer` | attention + couverture + mécanisme de copie (Pointer-Generator) |

Étapes du notebook : données → prétraitement et tokenisation → vocabulaire → encodage numérique → modèles → entraînement (teacher forcing) → décodage greedy → évaluation → comparaison → fonction `improve_text`.

## 1. Imports et configuration

In [ ]:
import os, re, json, time, math, random, copy, unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import Dataset, DataLoader, Sampler
import matplotlib.pyplot as plt
import sacrebleu   # uniquement pour calculer BLEU et chrF

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() or 1))

DATA_DIR = Path("data")
OUT_DIR = Path("prowrite_outputs")
CKPT_DIR, PLOT_DIR, ART_DIR = OUT_DIR / "checkpoints", OUT_DIR / "plots", OUT_DIR / "artifacts"
for d in (CKPT_DIR, PLOT_DIR, ART_DIR):
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "embedding_dim": 128,
    "hidden_dim": 256,
    "num_layers": 1,
    "dropout": 0.2,
    "batch_size": 64,
    "learning_rate": 1e-3,      # meilleur de 1e-3 / 5e-4 / 3e-4 lors d'un petit test préliminaire (2 epochs, validation)
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
    "teacher_forcing_ratio": 0.8,
    "max_epochs": 30,
    "early_stopping_patience": 5,
    "lr_scheduler_factor": 0.5,
    "lr_scheduler_patience": 2,
    "min_token_freq": 2,
    "coverage_lambda": 1.0,           # D_Pointer seulement
    "oov_copy_augmentation": 0.15,    # D_Pointer seulement
    "block_repeated_tokens": True,    # au décodage : pas deux fois le même mot de suite
    "beam_widths": [3, 5],
    "length_penalty_alpha": 0.7,
}
MODEL_NAMES = ["A_LSTM", "B_BiLSTM", "C_Attention", "D_Pointer"]
print("PyTorch", torch.__version__, "| device:", DEVICE, "| threads:", torch.get_num_threads())

## 2. Chargement et vérification des données

Dataset v3 : 36 000 paires synthétiques (30 000 train, 3 000 validation, 3 000 test).
Le **train** sert à apprendre les poids et le vocabulaire, la **validation** à choisir le meilleur checkpoint (early stopping), et le **test** n'est utilisé qu'à la fin.

In [ ]:
raw = {s: pd.read_csv(DATA_DIR / f"{s}.csv", keep_default_na=False, dtype=str) for s in ["train", "validation", "test"]}

for s, df in raw.items():
    print(f"{s:<11} {len(df):>6} lignes | vides: {(df[['source_text', 'target_text']] == '').sum().sum()} "
          f"| paires en double: {df.duplicated(['source_text', 'target_text']).sum()} "
          f"| déjà corrects (source = cible): {(df.source_text == df.target_text).sum()}")

# aucune phrase source ou cible ne doit apparaître dans deux splits
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    common_src = len(set(raw[a].source_text) & set(raw[b].source_text))
    common_tgt = len(set(raw[a].target_text) & set(raw[b].target_text))
    print(f"{a} / {b}: sources communes = {common_src}, cibles communes = {common_tgt}")

print("\nExemples :")
for _, r in raw["train"].sample(4, random_state=SEED).iterrows():
    print(" ", r.source_text, "\n   →", r.target_text)

## 3. Prétraitement : normalisation, ponctuation et tokenisation

* **Normalisation** : Unicode NFKC, guillemets et tirets typographiques remplacés, caractères invisibles supprimés, espaces fusionnés. On garde la casse (les majuscules font partie de ce que le modèle doit apprendre).
* **Ponctuation** : chaque signe est un token séparé, sauf dans les cas où il fait partie du mot (`a.m.`, `12:30`, `$1,500`, `Dr.`).
* **Tokenisation** : expression régulière écrite à la main. Les clitiques sont séparés (`don't` → `do` + `n't`), ce qui permet de traiter la négation comme un token.

`detokenize` fait l'opération inverse pour reconstruire une phrase à partir des tokens générés.

In [ ]:
TYPO = str.maketrans({"‘": "'", "’": "'", "‚": "'", "‛": "'", "′": "'", "“": '"', "”": '"', "„": '"', "″": '"',
                      "–": "-", "—": "-", "−": "-", " ": " "})
CONTROL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f​-‏⁠﻿]")

def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text)).translate(TYPO)
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace(" ", "\n").replace(" ", "\n")
    text = CONTROL.sub("", text)
    return re.sub(r"\s+", " ", text).strip()

TOKEN_PATTERN = r"""
    [ap]\.m\.                                  # a.m. / p.m.
  | (?:Dr|Mr|Mrs|Ms|Prof|St)\.(?=\s|$)         # titres
  | \d+[A-Z]\b                                 # codes comme 3B
  | \d+(?:[.,:]\d+)*                           # nombres : 9, 1,500, 12:30
  | [A-Za-z][A-Za-z0-9]*(?=n't\b)              # 'do' dans do|n't
  | n't\b
  | (?<=[A-Za-z])'(?:s|re|ll|ve|d|m)\b          # 's 're 'll 've 'd 'm
  | (?<=s)'(?![A-Za-z])                         # parents'
  | [A-Za-z][A-Za-z0-9]*(?:-[A-Za-z0-9]+)*     # mots
  | \.\.\.
  | [^\sA-Za-z0-9]                             # autres symboles
"""
TOKEN_RE = re.compile(TOKEN_PATTERN, re.VERBOSE)

def tokenize(text):
    return TOKEN_RE.findall(normalize_text(text))

NO_SPACE_BEFORE = {".", ",", "!", "?", ";", ":", "%", ")", "...", "n't", "'s", "'re", "'ll", "'ve", "'d", "'m", "'"}
NO_SPACE_AFTER = {"$", "#", "("}

def detokenize(tokens):
    out, prev, in_quote = "", None, False
    for tok in tokens:
        if tok == '"':
            out += '"' if in_quote else (" " if out else "") + '"'
            prev = '"close' if in_quote else '"open'
            in_quote = not in_quote
            continue
        if not out:
            out = tok
        elif tok in NO_SPACE_BEFORE or prev in NO_SPACE_AFTER or prev == '"open':
            out += tok
        else:
            out += " " + tok
        prev = tok
    return out

def finalize_text(text):
    # majuscule en début de phrase et pour le pronom I
    text = re.sub(r"\bi\b", "I", text)
    return re.sub(r"(^|[.!?]\s+)([a-z])", lambda m: m.group(1) + m.group(2).upper(), text)

print(normalize_text("  hey\tboss…\r\ni can’t  come “tomorrow”  "))
for s in ["hey can u send the report tomorrow", "Hi Sarah, I don't have the Q3 report... it's due at 3:00 p.m.?"]:
    print(tokenize(s))

# test aller-retour sur toutes les cibles du train
fails = [t for t in raw["train"].target_text if detokenize(tokenize(t)) != normalize_text(t)]
print(f"\ntokenize → detokenize identique sur {1 - len(fails) / len(raw['train']):.2%} des cibles du train")

In [ ]:
data = {}
for s, df in raw.items():
    d = df.copy()
    d["source_norm"] = d.source_text.map(normalize_text)
    d["target_norm"] = d.target_text.map(normalize_text)
    d["src_tokens"] = d.source_norm.map(tokenize)
    d["tgt_tokens"] = d.target_norm.map(tokenize)
    data[s] = d.reset_index(drop=True)

## 4. Vocabulaire et tokens spéciaux

Le vocabulaire est construit **uniquement sur le train** (sources + cibles). Un token vu moins de 2 fois est remplacé par `<UNK>`.

| ID | Token | Rôle |
|---|---|---|
| 0 | `<PAD>` | remplissage des séquences d'un batch, ignoré par la loss |
| 1 | `<UNK>` | mot inconnu du vocabulaire |
| 2 | `<SOS>` | premier input du décodeur |
| 3 | `<EOS>` | fin de séquence, arrête la génération |

In [ ]:
class Vocabulary:
    PAD, UNK, SOS, EOS = "<PAD>", "<UNK>", "<SOS>", "<EOS>"
    SPECIALS = [PAD, UNK, SOS, EOS]

    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.freq = Counter()
        self.itos = list(self.SPECIALS)
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    def build(self, token_lists):
        for toks in token_lists:
            self.freq.update(toks)
        words = [w for w, c in self.freq.items() if c >= self.min_freq and w not in self.SPECIALS]
        words.sort(key=lambda w: (-self.freq[w], w))
        self.itos = self.SPECIALS + words
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        return self

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        return [self.stoi.get(t, UNK_ID) for t in tokens]

    def decode(self, ids, oovs=None):
        toks = []
        for i in ids:
            i = int(i)
            if i == EOS_ID:
                break
            if i >= len(self.itos):      # mot copié hors vocabulaire (D_Pointer)
                k = i - len(self.itos)
                toks.append(oovs[k] if oovs and k < len(oovs) else self.UNK)
            elif self.itos[i] not in (self.PAD, self.SOS, self.EOS):
                toks.append(self.itos[i])
        return toks

    def to_dict(self):
        return {"min_freq": self.min_freq, "itos": self.itos, "freq": {t: self.freq[t] for t in self.itos if t in self.freq}}

PAD_ID, UNK_ID, SOS_ID, EOS_ID = 0, 1, 2, 3
vocab = Vocabulary(CONFIG["min_token_freq"]).build(list(data["train"].src_tokens) + list(data["train"].tgt_tokens))
VOCAB_SIZE = len(vocab)
assert [vocab.stoi[t] for t in Vocabulary.SPECIALS] == [PAD_ID, UNK_ID, SOS_ID, EOS_ID]
print("types de tokens dans le train :", len(vocab.freq), "| taille du vocabulaire (freq ≥ 2, + 4 spéciaux) :", VOCAB_SIZE)
print("tokens les plus fréquents :", vocab.itos[4:20])

# sauvegarde pour l'application
TOKENIZER_SPEC = {"type": "regex_word_tokenizer", "pattern": TOKEN_PATTERN, "flags": "re.VERBOSE",
                  "normalization": "NFKC + typographic map + control-char removal + whitespace collapse; case preserved",
                  "no_space_before": sorted(NO_SPACE_BEFORE), "no_space_after": sorted(NO_SPACE_AFTER)}
json.dump(vocab.to_dict(), open(ART_DIR / "vocab.json", "w", encoding="utf-8"), ensure_ascii=False)
json.dump(TOKENIZER_SPEC, open(ART_DIR / "tokenizer.json", "w", encoding="utf-8"), indent=1)

for text in ["hey boss i cant come tomorrow im sick sorry", "pls send the zorblaxian report asap"]:
    toks = tokenize(text)
    print("\n", text, "\n tokens :", toks, "\n IDs    :", vocab.encode(toks))

## 5. Représentation numérique : longueurs, troncature et padding

* source = tokens + `<EOS>` ; cible = `<SOS>` + tokens + `<EOS>`.
* Longueur maximale : 99,9e percentile du train arrondi au multiple de 8. Au-delà on tronque, en gardant `<EOS>`.
* Dans un batch, les séquences plus courtes sont complétées avec `<PAD>` ; un masque indique les vraies positions.

In [ ]:
src_len = data["train"].src_tokens.str.len() + 1
tgt_len = data["train"].tgt_tokens.str.len() + 2
MAX_SRC_LEN = int(8 * math.ceil(np.quantile(src_len, 0.999) / 8))
MAX_TGT_LEN = int(8 * math.ceil(np.quantile(tgt_len, 0.999) / 8))
CONFIG["max_source_length"], CONFIG["max_target_length"] = MAX_SRC_LEN, MAX_TGT_LEN
print("longueur moyenne source / cible :", round(src_len.mean(), 1), "/", round(tgt_len.mean(), 1),
      "| max retenu :", MAX_SRC_LEN, "/", MAX_TGT_LEN)

def encode_source(tokens, max_len=None):
    max_len = max_len or MAX_SRC_LEN
    return vocab.encode(tokens[: max_len - 1]) + [EOS_ID]

def encode_target(tokens, max_len=None):
    max_len = max_len or MAX_TGT_LEN
    return [SOS_ID] + vocab.encode(tokens[: max_len - 2]) + [EOS_ID]

long_toks = tokenize("could you please send me the final version of the report before friday afternoon")
print("\ntroncature à 8 :", [vocab.itos[i] for i in encode_source(long_toks, 8)])

a = encode_target(tokenize("Thank you."))
b = encode_target(tokenize("Please send the report today."))
padded = pad_sequence([torch.tensor(a), torch.tensor(b)], batch_first=True, padding_value=PAD_ID)
print("padding :\n", padded, "\nmasque :\n", padded != PAD_ID)

## 6. Dataset et DataLoader

Pour `D_Pointer`, chaque exemple a aussi des **IDs étendus** : un mot de la source absent du vocabulaire reçoit un ID temporaire (≥ taille du vocabulaire), ce qui permet au modèle de le recopier.

Pendant l'entraînement de `D_Pointer`, certains mots rares présents dans la source et la cible sont traités comme inconnus avec une probabilité de 0,15 (« OOV augmentation »), pour qu'il apprenne à recopier des mots jamais vus.

Les batchs regroupent des phrases de longueurs proches (moins de padding).

In [ ]:
def source_extended_ids(tokens, max_len=None):
    max_len = max_len or MAX_SRC_LEN
    ext, oovs = [], []
    for t in tokens[: max_len - 1]:
        i = vocab.stoi.get(t)
        if i is None:
            if t not in oovs:
                oovs.append(t)
            i = VOCAB_SIZE + oovs.index(t)
        ext.append(i)
    return ext + [EOS_ID], oovs

def target_extended_ids(tokens, oovs):
    ext = []
    for t in tokens[: MAX_TGT_LEN - 2]:
        i = vocab.stoi.get(t)
        if i is None:
            i = VOCAB_SIZE + oovs.index(t) if t in oovs else UNK_ID
        ext.append(i)
    return [SOS_ID] + ext + [EOS_ID]

class ProwriteDataset(Dataset):
    def __init__(self, df):
        self.examples = []
        for idx, r in enumerate(df.itertuples(index=False)):
            src_ext, oovs = source_extended_ids(r.src_tokens)
            self.examples.append({
                "index": idx, "unique_id": r.unique_id, "source": r.source_norm, "target": r.target_norm,
                "src": encode_source(r.src_tokens), "tgt": encode_target(r.tgt_tokens),
                "src_ext": src_ext, "tgt_ext": target_extended_ids(r.tgt_tokens, oovs), "oovs": oovs,
                "src_tok": r.src_tokens[: MAX_SRC_LEN - 1], "tgt_tok": r.tgt_tokens[: MAX_TGT_LEN - 2]})
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, i):
        return self.examples[i]

def with_forced_oov(e, forced):
    # reconstruit l'exemple comme si les mots de `forced` n'étaient pas dans le vocabulaire
    oovs, src, src_ext = [], [], []
    for t in e["src_tok"]:
        i = None if t in forced else vocab.stoi.get(t)
        if i is None:
            if t not in oovs:
                oovs.append(t)
            src.append(UNK_ID); src_ext.append(VOCAB_SIZE + oovs.index(t))
        else:
            src.append(i); src_ext.append(i)
    tgt, tgt_ext = [SOS_ID], [SOS_ID]
    for t in e["tgt_tok"]:
        i = None if t in forced else vocab.stoi.get(t)
        if i is None:
            tgt.append(UNK_ID); tgt_ext.append(VOCAB_SIZE + oovs.index(t) if t in oovs else UNK_ID)
        else:
            tgt.append(i); tgt_ext.append(i)
    return {**e, "src": src + [EOS_ID], "src_ext": src_ext + [EOS_ID], "tgt": tgt + [EOS_ID], "tgt_ext": tgt_ext + [EOS_ID], "oovs": oovs}

def augment_oov(batch, p, rng):
    out = []
    for e in batch:
        tgt_set = set(e["tgt_tok"])
        cands = {t for t in e["src_tok"] if t.isalpha() and t in tgt_set and vocab.freq.get(t, 0) < 300}
        forced = {t for t in sorted(cands) if rng.random() < p}
        out.append(with_forced_oov(e, forced) if forced else e)
    return out

def collate_batch(batch, oov_aug=0.0, rng=None):
    if oov_aug > 0 and rng is not None:
        batch = augment_oov(batch, oov_aug, rng)
    pad = lambda key: pad_sequence([torch.tensor(b[key]) for b in batch], batch_first=True, padding_value=PAD_ID)
    src, tgt = pad("src"), pad("tgt")
    return {"src": src, "tgt": tgt, "src_ext": pad("src_ext"), "tgt_ext": pad("tgt_ext"),
            "src_lengths": torch.tensor([len(b["src"]) for b in batch]), "src_mask": src != PAD_ID,
            "max_oovs": max(len(b["oovs"]) for b in batch), "oovs": [b["oovs"] for b in batch],
            "index": [b["index"] for b in batch]}

class BucketBatchSampler(Sampler):
    def __init__(self, lengths, batch_size, shuffle=True, chunk_batches=50, seed=SEED):
        self.lengths, self.bs, self.shuffle, self.chunk, self.seed, self.epoch = list(lengths), batch_size, shuffle, chunk_batches, seed, 0
    def __iter__(self):
        rng = random.Random(self.seed + self.epoch); self.epoch += 1
        idx = list(range(len(self.lengths)))
        if self.shuffle:
            rng.shuffle(idx)
        size = self.bs * self.chunk if self.shuffle else len(idx)
        batches = []
        for s in range(0, len(idx), size):
            chunk = sorted(idx[s:s + size], key=lambda j: self.lengths[j])
            batches += [chunk[k:k + self.bs] for k in range(0, len(chunk), self.bs)]
        if self.shuffle:
            rng.shuffle(batches)
        return iter(batches)
    def __len__(self):
        return math.ceil(len(self.lengths) / self.bs)

def make_loader(dataset, shuffle, oov_aug=0.0):
    lengths = [len(e["tgt"]) for e in dataset.examples]
    rng = random.Random(SEED + 7)
    collate = (lambda b: collate_batch(b, oov_aug, rng)) if oov_aug > 0 else collate_batch
    return DataLoader(dataset, batch_sampler=BucketBatchSampler(lengths, CONFIG["batch_size"], shuffle), collate_fn=collate)

def to_device(batch):
    return {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}

datasets = {s: ProwriteDataset(d) for s, d in data.items()}
loaders = {"validation": make_loader(datasets["validation"], shuffle=False),
           "test": make_loader(datasets["test"], shuffle=False)}

ex = next(e for e in datasets["test"].examples if e["oovs"])
print("source  :", ex["source"], "\nhors vocabulaire :", ex["oovs"])
print("src     :", ex["src"], "\nsrc_ext :", ex["src_ext"])

## 7. Les quatre architectures Encoder–Decoder

* **Encodeur** : embedding (128) → LSTM (256), ou BiLSTM (2 × 256) pour B, C et D. Il renvoie un état caché par mot et l'état final (h, c).
* **Décodeur** : LSTM (256) qui génère un token à la fois. À chaque pas : embedding du token précédent → LSTM → couche linéaire → softmax sur le vocabulaire.
* **B** : un « pont » (Linear + tanh) transforme les deux états finaux du BiLSTM en état initial du décodeur.
* **C** : attention de Bahdanau. Score e_j = vᵀ tanh(W_h h_j + W_s s_{t-1}), poids α = softmax(e), contexte c_t = Σ α_j h_j. Le décodeur reçoit [embedding ; contexte].
* **D** : en plus, une **couverture** (somme des attentions précédentes, pour éviter de regarder deux fois le même mot) et la **copie** :
  P(w) = p_gen · P_vocab(w) + (1 − p_gen) · Σ_{j : x_j = w} α_j

La boucle sur les pas de temps (`forward`) contient le **teacher forcing** : avec une probabilité de 0,8, l'entrée suivante est le vrai token de la cible, sinon c'est le token prédit par le modèle.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, embedding, emb_dim, hid_dim, num_layers, dropout, bidirectional):
        super().__init__()
        self.embedding = embedding
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.LSTM(emb_dim, hid_dim, num_layers=num_layers, batch_first=True, bidirectional=bidirectional,
                           dropout=dropout if num_layers > 1 else 0.0)

    def forward(self, src, src_lengths):
        embedded = self.dropout(self.embedding(src))
        packed = pack_padded_sequence(embedded, src_lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (h, c) = self.rnn(packed)
        outputs, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=src.size(1))
        return outputs, (h, c)


class Bridge(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.h_proj, self.c_proj = nn.Linear(2 * hid_dim, hid_dim), nn.Linear(2 * hid_dim, hid_dim)

    def forward(self, h, c):
        L2, B, H = h.shape
        cat = lambda x: x.view(L2 // 2, 2, B, H).permute(0, 2, 1, 3).reshape(L2 // 2, B, 2 * H)   # [avant ; arrière]
        return torch.tanh(self.h_proj(cat(h))), self.c_proj(cat(c))


class BahdanauAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim, attn_dim):
        super().__init__()
        self.W_h = nn.Linear(enc_dim, attn_dim, bias=False)
        self.W_s = nn.Linear(dec_dim, attn_dim, bias=True)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def project_encoder(self, enc_out):
        return self.W_h(enc_out)

    def forward(self, s_prev, enc_out, enc_proj, mask):
        scores = self.v(torch.tanh(enc_proj + self.W_s(s_prev).unsqueeze(1))).squeeze(-1)
        weights = torch.softmax(scores.masked_fill(~mask, float("-inf")), dim=-1)
        context = torch.bmm(weights.unsqueeze(1), enc_out).squeeze(1)
        return context, weights


class Seq2SeqBase(nn.Module):
    uses_copy = False

    def __init__(self, vocab_size, emb_dim, hid_dim, num_layers, dropout):
        super().__init__()
        self.vocab_size, self.emb_dim, self.hid_dim, self.num_layers = vocab_size, emb_dim, hid_dim, num_layers
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.dropout = nn.Dropout(dropout)

    def to_input_ids(self, ids):
        # un mot copié (ID étendu) n'a pas d'embedding : on le redonne au décodeur comme <UNK>
        return ids.masked_fill(ids >= self.vocab_size, UNK_ID)

    def forward(self, batch, teacher_forcing_ratio=1.0, rng=None):
        tgt = batch["tgt"]
        enc = self.encode(batch["src"], batch["src_lengths"], batch["src_mask"])
        state, y = enc["state"], tgt[:, 0]                       # premier input : <SOS>
        outputs, cov_losses = [], []
        for t in range(1, tgt.size(1)):
            log_probs, state, info = self.decode_step(y, state, enc, batch["src_ext"], batch["max_oovs"])
            outputs.append(log_probs)
            if "cov_loss" in info:
                cov_losses.append(info["cov_loss"])
            # teacher forcing
            use_gold = teacher_forcing_ratio >= 1.0 or (rng is not None and rng.random() < teacher_forcing_ratio)
            y = tgt[:, t] if use_gold else self.to_input_ids(log_probs.argmax(-1)).detach()
        self.last_coverage_loss = torch.stack(cov_losses, dim=1) if cov_losses else None
        return torch.stack(outputs, dim=1)


class A_LSTM(Seq2SeqBase):
    description = "Embedding → LSTM encoder → final (h,c) → LSTM decoder → Linear"

    def __init__(self, vocab_size, emb_dim, hid_dim, num_layers, dropout):
        super().__init__(vocab_size, emb_dim, hid_dim, num_layers, dropout)
        self.encoder = Encoder(self.embedding, emb_dim, hid_dim, num_layers, dropout, bidirectional=False)
        self.decoder_rnn = nn.LSTM(emb_dim, hid_dim, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.out = nn.Linear(hid_dim, vocab_size)

    def encode(self, src, src_lengths, src_mask):
        enc_out, (h, c) = self.encoder(src, src_lengths)
        return {"enc_out": enc_out, "mask": src_mask, "state": (h, c)}

    def decode_step(self, y_prev, state, enc, src_ext=None, n_oov=0):
        x = self.dropout(self.embedding(y_prev)).unsqueeze(1)
        out, state = self.decoder_rnn(x, state)
        logits = self.out(self.dropout(out.squeeze(1)))
        return F.log_softmax(logits, dim=-1), state, {}


class B_BiLSTM(A_LSTM):
    description = "Embedding → BiLSTM encoder → bridge([h_fwd;h_bwd]) → LSTM decoder → Linear"

    def __init__(self, vocab_size, emb_dim, hid_dim, num_layers, dropout):
        super().__init__(vocab_size, emb_dim, hid_dim, num_layers, dropout)
        self.encoder = Encoder(self.embedding, emb_dim, hid_dim, num_layers, dropout, bidirectional=True)
        self.bridge = Bridge(hid_dim)

    def encode(self, src, src_lengths, src_mask):
        enc_out, (h, c) = self.encoder(src, src_lengths)
        return {"enc_out": enc_out, "mask": src_mask, "state": self.bridge(h, c)}


class C_Attention(Seq2SeqBase):
    description = "Embedding → BiLSTM encoder → Bahdanau attention → LSTM decoder([x;context]) → Linear"

    def __init__(self, vocab_size, emb_dim, hid_dim, num_layers, dropout):
        super().__init__(vocab_size, emb_dim, hid_dim, num_layers, dropout)
        self.encoder = Encoder(self.embedding, emb_dim, hid_dim, num_layers, dropout, bidirectional=True)
        self.bridge = Bridge(hid_dim)
        self.attention = BahdanauAttention(2 * hid_dim, hid_dim, hid_dim)
        self.decoder_rnn = nn.LSTM(emb_dim + 2 * hid_dim, hid_dim, num_layers, batch_first=True,
                                   dropout=dropout if num_layers > 1 else 0.0)
        self.pre_out = nn.Linear(hid_dim + 2 * hid_dim + emb_dim, hid_dim)
        self.out = nn.Linear(hid_dim, vocab_size)

    def encode(self, src, src_lengths, src_mask):
        enc_out, (h, c) = self.encoder(src, src_lengths)
        return {"enc_out": enc_out, "enc_proj": self.attention.project_encoder(enc_out), "mask": src_mask,
                "state": self.bridge(h, c)}

    def decode_step(self, y_prev, state, enc, src_ext=None, n_oov=0):
        x = self.dropout(self.embedding(y_prev))
        context, attn = self.attention(state[0][-1], enc["enc_out"], enc["enc_proj"], enc["mask"])
        out, state = self.decoder_rnn(torch.cat([x, context], -1).unsqueeze(1), state)
        s_t = out.squeeze(1)
        logits = self.out(self.dropout(torch.tanh(self.pre_out(torch.cat([s_t, context, x], -1)))))
        return F.log_softmax(logits, dim=-1), state, {"attn": attn}


class CoverageAttention(BahdanauAttention):
    def __init__(self, enc_dim, dec_dim, attn_dim):
        super().__init__(enc_dim, dec_dim, attn_dim)
        self.W_u = nn.Linear(1, attn_dim, bias=False)

    def forward(self, s_prev, enc_out, enc_proj, mask, coverage):
        scores = self.v(torch.tanh(enc_proj + self.W_s(s_prev).unsqueeze(1) + self.W_u(coverage.unsqueeze(-1)))).squeeze(-1)
        weights = torch.softmax(scores.masked_fill(~mask, float("-inf")), dim=-1)
        context = torch.bmm(weights.unsqueeze(1), enc_out).squeeze(1)
        return context, weights


class D_Pointer(C_Attention):
    description = "BiLSTM + coverage attention + pointer/copy: P = p_gen·P_vocab + (1−p_gen)·Σ attention (extended vocabulary)"
    uses_copy = True

    def __init__(self, vocab_size, emb_dim, hid_dim, num_layers, dropout):
        super().__init__(vocab_size, emb_dim, hid_dim, num_layers, dropout)
        self.attention = CoverageAttention(2 * hid_dim, hid_dim, hid_dim)
        self.p_gen_linear = nn.Linear(2 * hid_dim + hid_dim + emb_dim, 1)

    def encode(self, src, src_lengths, src_mask):
        enc = super().encode(src, src_lengths, src_mask)
        h, c = enc["state"]
        enc["state"] = (h, c, torch.zeros(src.size(0), src.size(1), device=src.device))   # couverture initiale = 0
        return enc

    def decode_step(self, y_prev, state, enc, src_ext=None, n_oov=0):
        h, c, cov = state
        x = self.dropout(self.embedding(y_prev))
        context, attn = self.attention(h[-1], enc["enc_out"], enc["enc_proj"], enc["mask"], cov)
        out, (h, c) = self.decoder_rnn(torch.cat([x, context], -1).unsqueeze(1), (h, c))
        s_t = out.squeeze(1)
        logits = self.out(self.dropout(torch.tanh(self.pre_out(torch.cat([s_t, context, x], -1)))))
        p_gen = torch.sigmoid(self.p_gen_linear(torch.cat([context, s_t, x], -1)))
        cov_loss = torch.minimum(attn, cov).sum(-1)
        vocab_dist = torch.softmax(logits, dim=-1)
        if n_oov > 0:
            vocab_dist = torch.cat([vocab_dist, vocab_dist.new_zeros(vocab_dist.size(0), n_oov)], dim=-1)
        final = (p_gen * vocab_dist).scatter_add(1, src_ext, (1 - p_gen) * attn)       # générer ou copier
        info = {"attn": attn, "p_gen": p_gen, "cov_loss": cov_loss}
        return torch.log(final.clamp_min(1e-12)), (h, c, cov + attn), info


MODEL_CLASSES = {"A_LSTM": A_LSTM, "B_BiLSTM": B_BiLSTM, "C_Attention": C_Attention, "D_Pointer": D_Pointer}

def build_model(name):
    return MODEL_CLASSES[name](VOCAB_SIZE, CONFIG["embedding_dim"], CONFIG["hidden_dim"], CONFIG["num_layers"], CONFIG["dropout"]).to(DEVICE)

pd.DataFrame([{"modèle": n, "paramètres": sum(p.numel() for p in build_model(n).parameters())} for n in MODEL_NAMES]).set_index("modèle")

## 8. Fonction de perte et boucle d'entraînement

* Loss : **NLL** (log-vraisemblance négative = entropie croisée) moyenne par token, les `<PAD>` sont ignorés. Pour `D_Pointer` on ajoute la loss de couverture (λ = 1).
* Optimiseur **AdamW**, gradient clipping à 1,0.
* Après chaque epoch : loss de validation (sans dropout, teacher forcing = 1). `ReduceLROnPlateau` divise le learning rate par 2 si la validation ne progresse pas pendant 2 epochs.
* **Early stopping** après 5 epochs sans amélioration. Le meilleur checkpoint (plus faible loss de validation) est sauvegardé puis rechargé à la fin.

In [ ]:
def sequence_loss(model, batch, teacher_forcing_ratio=1.0, rng=None, with_coverage=True):
    log_probs = model(batch, teacher_forcing_ratio, rng)
    gold = (batch["tgt_ext"] if model.uses_copy else batch["tgt"])[:, 1:]
    loss = F.nll_loss(log_probs.reshape(-1, log_probs.size(-1)), gold.reshape(-1), ignore_index=PAD_ID)
    if with_coverage and model.last_coverage_loss is not None:
        m = (gold != PAD_ID).float()
        loss = loss + CONFIG["coverage_lambda"] * (model.last_coverage_loss * m).sum() / m.sum()
    n_tok = int((gold != PAD_ID).sum())
    correct = int(((log_probs.argmax(-1) == gold) & (gold != PAD_ID)).sum())
    return loss, n_tok, correct

def train_one_epoch(model, loader, optimizer, rng):
    model.train()
    total, tokens = 0.0, 0
    for batch in loader:
        batch = to_device(batch)
        optimizer.zero_grad(set_to_none=True)
        loss, n_tok, _ = sequence_loss(model, batch, CONFIG["teacher_forcing_ratio"], rng)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        optimizer.step()
        total += loss.item() * n_tok; tokens += n_tok
    return total / tokens

@torch.no_grad()
def evaluate_loss(model, loader):
    model.eval()
    total, tokens, correct = 0.0, 0, 0
    for batch in loader:
        loss, n_tok, c = sequence_loss(model, to_device(batch), 1.0, with_coverage=False)
        total += loss.item() * n_tok; tokens += n_tok; correct += c
    return total / tokens, correct / tokens

def save_checkpoint(model, name, path, extra):
    torch.save({"model_name": name, "architecture": model.description, "model_class": type(model).__name__,
                "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "model_config": {"vocab_size": VOCAB_SIZE, "embedding_dim": model.emb_dim, "hidden_dim": model.hid_dim,
                                 "num_layers": model.num_layers, "dropout": CONFIG["dropout"]},
                "hyperparameters": copy.deepcopy(CONFIG), "seed": SEED,
                "max_source_length": MAX_SRC_LEN, "max_target_length": MAX_TGT_LEN,
                "special_tokens": {"<PAD>": PAD_ID, "<UNK>": UNK_ID, "<SOS>": SOS_ID, "<EOS>": EOS_ID},
                "vocab": vocab.to_dict(), "tokenizer": TOKENIZER_SPEC,
                "torch_version": torch.__version__, "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"), **extra}, path)

def load_checkpoint(path):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model = build_model(ckpt["model_name"])
    model.load_state_dict(ckpt["model_state_dict"])
    return model.eval(), ckpt

## 9. Entraînement des quatre modèles

Chaque modèle part d'une initialisation aléatoire (même graine), voit les données dans le même ordre et utilise les mêmes hyperparamètres.

In [ ]:
def fit(name):
    set_seed(SEED)
    model = build_model(name)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=CONFIG["lr_scheduler_factor"],
                                                           patience=CONFIG["lr_scheduler_patience"])
    train_loader = make_loader(datasets["train"], shuffle=True,
                               oov_aug=CONFIG["oov_copy_augmentation"] if model.uses_copy else 0.0)
    rng = random.Random(SEED)          # tirages du teacher forcing
    path = CKPT_DIR / f"{name}.pt"
    best_val, best_epoch, bad, history, stop = float("inf"), 0, 0, [], "max_epochs"
    t_start = time.time()
    for epoch in range(1, CONFIG["max_epochs"] + 1):
        t0 = time.time()
        lr = optimizer.param_groups[0]["lr"]
        train_loss = train_one_epoch(model, train_loader, optimizer, rng)
        val_loss, val_acc = evaluate_loss(model, loaders["validation"])
        scheduler.step(val_loss)
        improved = val_loss < best_val - 1e-4
        if improved:
            best_val, best_epoch, bad = val_loss, epoch, 0
        else:
            bad += 1
        history.append({"model": name, "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "val_token_acc": val_acc, "learning_rate": lr, "epoch_seconds": time.time() - t0})
        if improved:
            save_checkpoint(model, name, path, {"best_epoch": best_epoch, "best_val_loss": best_val})
        print(f"{name:<12} epoch {epoch:>2} | train {train_loss:.4f} | val {val_loss:.4f} | acc {val_acc:.3f} | "
              f"lr {lr:.0e} | {time.time() - t0:.0f}s {'*' if improved else ''}")
        if bad >= CONFIG["early_stopping_patience"]:
            stop = "early_stopping"
            break
    minutes = (time.time() - t_start) / 60
    model, ckpt = load_checkpoint(path)          # on garde les meilleurs poids
    ckpt.update({"training_history": history, "stop_reason": stop, "epochs_trained": len(history), "train_minutes": minutes})
    torch.save(ckpt, path)
    print(f"-> {name} : meilleur epoch {best_epoch}, val loss {best_val:.4f}, {len(history)} epochs, {minutes:.1f} min\n")
    return model, history, {"model": name, "epochs_trained": len(history), "best_epoch": best_epoch,
                            "best_val_loss": best_val, "stop_reason": stop, "train_minutes": minutes}

trained, histories, summaries = {}, [], []
for name in MODEL_NAMES:
    model, hist, summ = fit(name)
    trained[name] = model
    histories += hist
    summaries.append(summ)

history_df = pd.DataFrame(histories)
history_df.to_csv(OUT_DIR / "training_history.csv", index=False)
summary_df = pd.DataFrame(summaries).set_index("model")
summary_df.round(4)

## 10. Courbes d'apprentissage

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, name in zip(axes.ravel(), MODEL_NAMES):
    h = history_df[history_df.model == name]
    ax.plot(h.epoch, h.train_loss, "o-", label="train")
    ax.plot(h.epoch, h.val_loss, "s-", label="validation")
    ax.axvline(summary_df.loc[name, "best_epoch"], c="gray", ls=":", label="meilleur epoch")
    ax.set(title=name, xlabel="epoch", ylabel="NLL par token")
    ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(PLOT_DIR / "learning_curves_per_model.png", dpi=120); plt.show()

La loss d'entraînement continue de baisser alors que la loss de validation se stabilise vers l'epoch 3 à 5 : c'est le début du sur-apprentissage, d'où l'early stopping. A et B restent à une loss de validation élevée (≈ 3,8) : sans attention, ils n'arrivent pas à bien utiliser le message source. C et D descendent beaucoup plus bas.

Remarque : la loss d'entraînement est mesurée avec dropout et 80 % de teacher forcing (et la couverture pour D), donc elle n'est pas directement comparable à la loss de validation.

## 11. Décodage greedy (inférence)

À l'inférence il n'y a pas de cible : on part de `<SOS>`, on choisit à chaque pas le token le plus probable (argmax), on le redonne en entrée, et on s'arrête à `<EOS>` (ou à la longueur maximale).
Contrainte ajoutée : le même mot ne peut pas être produit deux fois de suite.

In [ ]:
WORD_IDS = torch.tensor([bool(re.fullmatch(r"[A-Za-z][A-Za-z0-9'-]*", t)) for t in vocab.itos], device=DEVICE)

def block_repeats(log_probs, prev_ids):
    if not CONFIG["block_repeated_tokens"]:
        return log_probs
    is_word = torch.where(prev_ids < VOCAB_SIZE, WORD_IDS[prev_ids.clamp(max=VOCAB_SIZE - 1)], torch.ones_like(prev_ids, dtype=torch.bool))
    is_word &= prev_ids > EOS_ID
    rows = is_word.nonzero().flatten()
    if len(rows):
        log_probs = log_probs.clone()
        log_probs[rows, prev_ids[rows]] = float("-inf")
    return log_probs

@torch.no_grad()
def greedy_decode_batch(model, batch):
    model.eval()
    enc = model.encode(batch["src"], batch["src_lengths"], batch["src_mask"])
    state = enc["state"]
    B = batch["src"].size(0)
    y = torch.full((B,), SOS_ID, dtype=torch.long, device=DEVICE)
    prev = y.clone()
    finished = torch.zeros(B, dtype=torch.bool, device=DEVICE)
    steps = []
    for _ in range(MAX_TGT_LEN - 1):
        log_probs, state, _ = model.decode_step(y, state, enc, batch["src_ext"], batch["max_oovs"])
        next_ids = block_repeats(log_probs, prev).argmax(-1).masked_fill(finished, PAD_ID)
        prev = next_ids
        steps.append(next_ids)
        finished |= next_ids == EOS_ID
        if finished.all():
            break
        y = model.to_input_ids(next_ids)
    ids = torch.stack(steps, dim=1).tolist()
    return [vocab.decode(row, oovs=batch["oovs"][i]) for i, row in enumerate(ids)]

def generate(model, split):
    hyps = [None] * len(datasets[split])
    for batch in loaders[split]:
        for idx, toks in zip(batch["index"], greedy_decode_batch(model, to_device(batch))):
            hyps[idx] = finalize_text(detokenize(toks))
    return hyps

@torch.no_grad()
def greedy_trace(model, example):
    # décodage pas à pas d'un seul message, pour voir chaque décision
    model.eval()
    b = to_device(collate_batch([example]))
    src_tokens = [vocab.itos[i] if i < VOCAB_SIZE else example["oovs"][i - VOCAB_SIZE] for i in example["src_ext"]]
    enc = model.encode(b["src"], b["src_lengths"], b["src_mask"])
    state, y, rows, prev = enc["state"], torch.tensor([SOS_ID], device=DEVICE), [], SOS_ID
    for t in range(1, MAX_TGT_LEN):
        lp, state, info = model.decode_step(y, state, enc, b["src_ext"], b["max_oovs"])
        p, nxt = block_repeats(lp, torch.tensor([prev], device=DEVICE)).exp().max(-1)
        nxt = int(nxt)
        tok = vocab.itos[nxt] if nxt < VOCAB_SIZE else example["oovs"][nxt - VOCAB_SIZE] + " (copié)"
        row = {"t": t, "entrée": vocab.itos[int(y)], "sortie": tok, "P": round(float(p), 3)}
        if "attn" in info:
            j = int(info["attn"][0].argmax())
            row["mot le plus regardé"] = f"{src_tokens[j]} ({float(info['attn'][0, j]):.2f})"
        if "p_gen" in info:
            row["p_gen"] = round(float(info["p_gen"]), 3)
        rows.append(row)
        if nxt == EOS_ID:
            break
        prev = nxt
        y = model.to_input_ids(torch.tensor([nxt], device=DEVICE))
    return pd.DataFrame(rows).set_index("t")

example = next(e for e in datasets["test"].examples if e["oovs"] and any(i >= VOCAB_SIZE for i in e["tgt_ext"]) and len(e["tgt"]) < 30)
print("message   :", example["source"])
print("référence :", example["target"], "(non donnée au modèle)")
greedy_trace(trained["D_Pointer"], example)

In [ ]:
b = to_device(collate_batch([example]))
for name in MODEL_NAMES:
    print(f"{name:<12}: {finalize_text(detokenize(greedy_decode_batch(trained[name], b)[0]))}")

## 12. Beam search (comparaison)

Le beam search garde les k meilleures hypothèses à chaque pas (score = somme des log-probabilités, divisé par longueur^0,7). On le compare au greedy sur 150 messages du test.

In [ ]:
def select_state(state, idx):
    return tuple(t[:, idx] if t.dim() == 3 else t[idx] for t in state)

@torch.no_grad()
def beam_search_decode(model, example, beam_width=3):
    model.eval()
    alpha = CONFIG["length_penalty_alpha"]
    b = to_device(collate_batch([example]))
    enc = model.encode(b["src"], b["src_lengths"], b["src_mask"])
    state, seqs, scores, finished = enc["state"], [[]], torch.zeros(1, device=DEVICE), []
    for _ in range(MAX_TGT_LEN - 1):
        k = len(seqs)
        enc_k = {key: val.expand(k, *val.shape[1:]) for key, val in enc.items() if key != "state"}
        last = torch.tensor([s[-1] if s else SOS_ID for s in seqs], device=DEVICE)
        log_probs, new_state, _ = model.decode_step(model.to_input_ids(last), state, enc_k, b["src_ext"].expand(k, -1), b["max_oovs"])
        log_probs = block_repeats(log_probs, last)
        cand = (scores.unsqueeze(1) + log_probs).view(-1)
        top_scores, top_idx = cand.topk(min(2 * beam_width, cand.numel()))
        V = log_probs.size(1)
        new_seqs, new_scores, origin = [], [], []
        for sc, ix in zip(top_scores.tolist(), top_idx.tolist()):
            beam, tok = divmod(ix, V)
            if tok == EOS_ID:
                finished.append((seqs[beam] + [tok], sc / ((len(seqs[beam]) + 1) ** alpha)))
            else:
                new_seqs.append(seqs[beam] + [tok]); new_scores.append(sc); origin.append(beam)
            if len(new_seqs) == beam_width:
                break
        if len(finished) >= beam_width or not new_seqs:
            break
        seqs, scores = new_seqs, torch.tensor(new_scores, device=DEVICE)
        state = select_state(new_state, torch.tensor(origin, device=DEVICE))
    if not finished:
        finished = [(s, sc / (len(s) ** alpha)) for s, sc in zip(seqs, scores.tolist())]
    best = max(finished, key=lambda f: f[1])
    return vocab.decode(best[0], oovs=example["oovs"])

beam_examples = [datasets["test"][i] for i in random.Random(SEED).sample(range(len(datasets["test"])), 150)]
beam_refs = [e["target"] for e in beam_examples]
rows = []
for name in MODEL_NAMES:
    t0 = time.time()
    hyp = [finalize_text(detokenize(greedy_decode_batch(trained[name], to_device(collate_batch([e])))[0])) for e in beam_examples]
    rows.append({"model": name, "decoding": "greedy", "BLEU": sacrebleu.corpus_bleu(hyp, [beam_refs]).score,
                 "chrF": sacrebleu.corpus_chrf(hyp, [beam_refs]).score, "ms_per_sentence": (time.time() - t0) / 0.15})
    for k in CONFIG["beam_widths"]:
        t0 = time.time()
        hyp = [finalize_text(detokenize(beam_search_decode(trained[name], e, k))) for e in beam_examples]
        rows.append({"model": name, "decoding": f"beam k={k}", "BLEU": sacrebleu.corpus_bleu(hyp, [beam_refs]).score,
                     "chrF": sacrebleu.corpus_chrf(hyp, [beam_refs]).score, "ms_per_sentence": (time.time() - t0) / 0.15})
beam_df = pd.DataFrame(rows)
beam_df.to_csv(OUT_DIR / "beam_vs_greedy_test_subset.csv", index=False)
beam_df.round(2).set_index(["model", "decoding"])

## 13. Évaluation quantitative sur le test

Toutes les sorties sont générées en greedy. Métriques :

* **BLEU** et **chrF** (sacreBLEU), **ROUGE-L** (F1 sur la plus longue sous-séquence commune), **exact match** ;
* **fidélité des faits** : part des noms propres et nombres de la référence (présents dans la source) retrouvés dans la sortie ;
* **nombres inventés** : sorties contenant un nombre absent de la source ;
* **négation** : la sortie et la référence contiennent (ou non) toutes les deux une négation ;
* **copie OOV** : quand la référence contient un mot hors vocabulaire venant de la source, est-il dans la sortie ?
* **latence** : temps moyen pour un message (batch de 1, CPU).

Référence simple : renvoyer le message tel quel (« copy_input_baseline »).

In [ ]:
NEG_RE = re.compile(r"\b(not|no|never|cannot|unable|without|nobody|nothing|none|neither|nor)\b|n't\b", re.I)
NUM_RE = r"\d+(?:[.,:]\d+)*"

def rouge_l_f1(h, r):
    if not h or not r:
        return 0.0
    prev = [0] * (len(r) + 1)
    for a in h:
        cur = [0]
        for j, bb in enumerate(r):
            cur.append(prev[j] + 1 if a == bb else max(prev[j + 1], cur[j]))
        prev = cur
    lcs = prev[-1]
    return 0.0 if lcs == 0 else 2 * lcs / (len(h) + len(r))

def fact_tokens(ref, src):
    rt, st = tokenize(ref), {t.lower() for t in tokenize(src)}
    src_nums = set(re.findall(NUM_RE, src))
    facts = []
    for i, t in enumerate(rt):
        if re.fullmatch(NUM_RE, t) and t in src_nums:
            facts.append(t)
        elif t[0].isupper() and t != "I" and i > 0 and rt[i - 1] not in {".", "?", "!"} and t.lower() in st:
            facts.append(t)
    return facts

def score_outputs(hyps, split):
    ex = datasets[split].examples
    refs, srcs = [e["target"] for e in ex], [e["source"] for e in ex]
    htoks = [tokenize(h) for h in hyps]
    hit = tot = 0
    for h, r, s in zip(htoks, refs, srcs):
        hc = Counter(h)
        for tok, n in Counter(fact_tokens(r, s)).items():
            hit += min(n, hc[tok]); tot += n
    oov_cases = [(e, h) for e, h in zip(ex, htoks) if any(i >= VOCAB_SIZE for i in e["tgt_ext"])]
    return {"BLEU": sacrebleu.corpus_bleu(hyps, [refs]).score,
            "ROUGE_L": 100 * np.mean([rouge_l_f1(h, tokenize(r)) for h, r in zip(htoks, refs)]),
            "chrF": sacrebleu.corpus_chrf(hyps, [refs]).score,
            "exact_match_%": 100 * np.mean([h == r for h, r in zip(hyps, refs)]),
            "fact_recall_%": 100 * hit / max(tot, 1),
            "number_hallucination_%": 100 * np.mean([bool(set(re.findall(NUM_RE, h)) - set(re.findall(NUM_RE, s))) for h, s in zip(hyps, srcs)]),
            "negation_agreement_%": 100 * np.mean([bool(NEG_RE.search(h)) == bool(NEG_RE.search(r)) for h, r in zip(hyps, refs)]),
            "oov_copy_success_%": 100 * np.mean([all(e["oovs"][i - VOCAB_SIZE] in h for i in e["tgt_ext"] if i >= VOCAB_SIZE)
                                                 for e, h in oov_cases])}

predictions = {"copy_input_baseline": [e["source"] for e in datasets["test"].examples]}
for name in MODEL_NAMES:
    predictions[name] = generate(trained[name], "test")

# latence : 100 messages du test, un par un
lat_examples = [datasets["test"][i] for i in random.Random(SEED + 1).sample(range(len(datasets["test"])), 100)]
latency = {}
for name in MODEL_NAMES:
    greedy_decode_batch(trained[name], to_device(collate_batch([lat_examples[0]])))     # échauffement
    t0 = time.time()
    for e in lat_examples:
        greedy_decode_batch(trained[name], to_device(collate_batch([e])))
    latency[name] = 1000 * (time.time() - t0) / len(lat_examples)

rows = []
for name in ["copy_input_baseline"] + MODEL_NAMES:
    r = {"model": name, **score_outputs(predictions[name], "test")}
    if name in trained:
        r["val_loss"] = summary_df.loc[name, "best_val_loss"]
        r["test_loss"] = evaluate_loss(trained[name], loaders["test"])[0]
        r["latency_ms_per_message"] = latency[name]
    rows.append(r)
test_metrics = pd.DataFrame(rows).set_index("model")
test_metrics.to_csv(OUT_DIR / "test_metrics.csv")
test_metrics.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
test_metrics[["BLEU", "ROUGE_L", "chrF"]].plot.bar(ax=ax[0], rot=15, title="Métriques sur le test (greedy)")
test_metrics.loc[MODEL_NAMES, ["fact_recall_%", "negation_agreement_%", "oov_copy_success_%"]].plot.bar(ax=ax[1], rot=15, title="Préservation du sens (test)")
for a in ax:
    a.grid(alpha=.3, axis="y")
plt.tight_layout(); plt.savefig(PLOT_DIR / "test_metric_comparison.png", dpi=120); plt.show()

pd.DataFrame({"unique_id": [e["unique_id"] for e in datasets["test"].examples],
              "source_text": [e["source"] for e in datasets["test"].examples],
              "reference": [e["target"] for e in datasets["test"].examples],
              **{n: predictions[n] for n in MODEL_NAMES}}).to_csv(OUT_DIR / "test_predictions.csv", index=False)

## 14. Évaluation qualitative

Quelques messages du test (jamais vus pendant l'entraînement), avec la sortie des quatre modèles et le chrF de chaque phrase.

In [ ]:
for i in random.Random(3).sample(range(len(datasets["test"])), 6):
    e = datasets["test"][i]
    print(f"\nMESSAGE   : {e['source']}\nRÉFÉRENCE : {e['target']}")
    for name in MODEL_NAMES:
        h = predictions[name][i]
        print(f"  {name:<12} {h}   (chrF {sacrebleu.sentence_chrf(h, [e['target']]).score:.0f})")

On retrouve ce que montrent les chiffres : A et B produisent des phrases correctes en apparence mais inventent souvent le contenu (autre objet, autre nom). C garde la structure du message mais perd parfois les noms et nombres rares. D recopie ces mots, mais il lui arrive aussi de recopier une faute de la source ou de perdre une négation.

## 15. Comparaison des quatre modèles

In [ ]:
comparison = pd.DataFrame({
    "architecture": {n: MODEL_CLASSES[n].description for n in MODEL_NAMES},
    "paramètres": {n: sum(p.numel() for p in trained[n].parameters()) for n in MODEL_NAMES},
    "epochs": summary_df["epochs_trained"], "meilleur epoch": summary_df["best_epoch"],
    "minutes": summary_df["train_minutes"].round(1),
    "val loss": summary_df["best_val_loss"].round(3),
    "BLEU": test_metrics.loc[MODEL_NAMES, "BLEU"].round(1), "ROUGE-L": test_metrics.loc[MODEL_NAMES, "ROUGE_L"].round(1),
    "chrF": test_metrics.loc[MODEL_NAMES, "chrF"].round(1), "faits %": test_metrics.loc[MODEL_NAMES, "fact_recall_%"].round(1),
    "latence ms": test_metrics.loc[MODEL_NAMES, "latency_ms_per_message"].round(1)})
comparison.to_csv(OUT_DIR / "model_comparison.csv")
comparison

**Lecture des résultats**

* Renvoyer le message sans le modifier donne déjà un BLEU d'environ 42, car beaucoup de mots ne changent pas. A et B font **moins bien** que cette référence.
* L'attention (B → C) est le gain principal : le décodeur peut regarder chaque mot du message au lieu d'un seul vecteur résumé.
* La copie et la couverture (C → D) améliorent encore tous les scores, surtout la conservation des noms, des nombres et des mots hors vocabulaire. D n'a que ~1 000 paramètres de plus que C : le gain vient du mécanisme, pas de la taille.
* La latence augmente un peu avec la complexité du modèle mais reste de l'ordre de quelques dizaines de millisecondes par message sur CPU.

Le meilleur modèle selon la loss de validation est `D_Pointer` ; c'est celui utilisé par l'application.

## 16. Fonction `improve_text`

Pipeline complet pour un nouveau message : normalisation → tokenisation → IDs → modèle chargé depuis son checkpoint → décodage → texte.

In [ ]:
_cache = {}
def improve_text(text, model_name="D_Pointer", decoding="greedy", beam_width=3):
    if model_name not in _cache:
        _cache[model_name] = load_checkpoint(CKPT_DIR / f"{model_name}.pt")[0]
    model = _cache[model_name]
    norm = normalize_text(text)
    toks = tokenize(norm)
    src_ext, oovs = source_extended_ids(toks)
    example = {"index": 0, "src": encode_source(toks), "tgt": [SOS_ID, EOS_ID], "tgt_ext": [SOS_ID, EOS_ID],
               "src_ext": src_ext, "oovs": oovs}
    if decoding == "greedy":
        out = greedy_decode_batch(model, to_device(collate_batch([example])))[0]
    else:
        out = beam_search_decode(model, example, beam_width)
    return finalize_text(detokenize(out))

demo = ["hey boss i cant come tomorrow im sick sorry",
        "can u send me the report today pls i need it",
        "the meeting was yesterday but nobody tell me",
        "sorry i forgot to reply i was busy",
        "Please let me know when the report is ready.",
        "hi sarah my order #884213 still didnt arrive can u check pls"]
demo_df = pd.DataFrame([{"input": t, **{n: improve_text(t, n) for n in MODEL_NAMES}} for t in demo])
demo_df.to_csv(OUT_DIR / "improve_text_demo.csv", index=False)
for _, r in demo_df.iterrows():
    print(f"\n{r['input']}")
    for n in MODEL_NAMES:
        print(f"  {n:<12} {r[n]}")
print("\nbeam (k=3), D_Pointer :", improve_text(demo[0], decoding="beam"))

## 17. Limites

* **Données synthétiques** : générées à partir de modèles de phrases. Les scores valent pour cette distribution ; le modèle n'a pas été testé sur de vrais messages.
* **Sur-apprentissage** : la validation se stabilise dès l'epoch 3 à 5.
* **Mots rares** : la copie de `D_Pointer` ne marche pas toujours, et elle recopie aussi les fautes de la source (`didnt`, `pls`).
* **Sens** : il arrive que la négation soit perdue (« nobody tell me » → « it told me ») ou qu'un nombre soit ajouté.
* **A et B** ne sont pas utilisables en pratique.
* **Métriques** : BLEU, ROUGE-L et chrF comparent à une seule référence ; un bon score ne garantit pas une phrase parfaite. Une évaluation humaine serait nécessaire.
* Les hyperparamètres n'ont été que peu explorés (CPU).